# 🔁 Lab 05 — Recurrent Neural Networks (RNN) in PyTorch

**DL2026 · Practical Session 5 · A very simple, step-by-step introduction**

So far our networks saw the **whole input at once**:

* Lab 02 — a *list of 4 numbers* (iris measurements), then a *picture* flattened into 784 numbers.
* Lab 04 — a *picture*, using the fact that **neighbouring pixels belong together**.

This week the input is a **sequence**: numbers that arrive **one after another in time**, where the
**order itself is the information**. Sequences are everywhere:

| Sequence | One "time step" is… | A typical question |
|:--|:--|:--|
| temperature every hour | one reading | what will it be at 15:00? |
| a sentence | one word | is this review positive? |
| an ECG signal | one sample | is this heartbeat normal? |
| daily sales | one day | how many will we sell tomorrow? |

The tool for this is the **Recurrent Neural Network**: a network with a **memory**. It reads the
sequence one step at a time and keeps a small summary of everything it has seen so far — the
**hidden state**.

### Our example, on purpose the simplest possible

> **The data:** a **sine wave** — 400 numbers, generated in one line, nothing to download.
> **The task:** *given the last 20 values, predict the next one.*

Small enough to train in a few seconds on a CPU, and you can **see** whether it works by looking at a
plot. Everything you learn here transfers unchanged to text, sensors, audio and stock prices — only the
numbers get bigger.


## 🎯 What you will be able to do at the end

| # | You will be able to… | Step |
|:--|:---------------------|:--|
| 1 | Explain why order matters, and turn a raw signal into `(batch, seq_len, input_size)` tensors | [Step 1](#step1) |
| 2 | Compute an RNN's hidden state **by hand**, and get exactly the same numbers from `nn.RNN` | [Step 2](#step2) |
| 3 | Build a model: `nn.RNN` + `nn.Linear`, and say what `output` and `h_n` contain | [Step 3](#step3) |
| 4 | Pick the right loss for a sequence task | [Step 4](#step4) |
| 5 | Write the training loop (the same 4 lines as always) | [Step 5](#step5) |
| 6 | Use the trained model: one-step prediction **and** multi-step forecasting | [Step 6](#step6) |
| 7 | Swap `nn.RNN` → `nn.LSTM` / `nn.GRU` in one line, and say why you would | [Step 7](#step7) |
| 8 | Use an RNN for **classification** of whole sequences | [Step 8](#step8) |

## 📑 The 8 steps

| Step | Topic |
|:--|:--|
| [0](#step0) | Setup — imports, seed, device |
| [1](#step1) | The data — a sine wave, sliding windows, the 3-D tensor shape |
| [2](#step2) | What an RNN really does — one formula, by hand, then `nn.RNN` |
| [3](#step3) | The model — `nn.RNN` + `nn.Linear` |
| [4](#step4) | Loss function and optimizer |
| [5](#step5) | The training loop |
| [6](#step6) | Using the trained model (predict + forecast) |
| [7](#step7) | LSTM and GRU — a one-line upgrade |
| [8](#step8) | A second example — sequence **classification** |
| [9](#step9) | Exercises |
| [10](#step10) | Summary, cheat sheet, common bugs |

> 💡 **How to work through this notebook.** Run every cell in order, top to bottom. Read the short text
> *before* each cell, then look at the printed output and check it says what the text promised.


---
<a id="step0"></a>
# Step 0 · Setup

**Goal:** import everything once, fix the random seed, and choose the device.

Nothing here is new — it is the same opening cell as Lab 02 and Lab 04. Fixing `SEED` means you and the
person sitting next to you get the *same* numbers, which makes debugging together possible.

This notebook is tiny: **it runs perfectly well on a CPU**, no GPU required.


In [ ]:
import numpy as np
import torch
import torch.nn as nn                     # layers: RNN, LSTM, Linear, ...
import torch.optim as optim               # optimizers: Adam, SGD, ...
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# Fix the randomness so this notebook is reproducible.
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use the GPU if there is one. For this small notebook the CPU is perfectly fine.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("PyTorch version :", torch.__version__)
print("Device          :", DEVICE)
print("Setup done ✅")

---
<a id="step1"></a>
# Step 1 · The Data

**Goal:** build a tiny sequence dataset and get it into the shape PyTorch expects.

## 1.1 First: why "order" needs a new kind of network

Take two sequences that contain **exactly the same numbers** in a different order. Every statistic that
ignores order — sum, mean, max, the set of values — is *identical*. Only the **arrangement** tells them
apart: one goes up, the other goes down.

A feed-forward network *can* see position (input 1 and input 5 have different weights), but it needs a
**fixed-length** input and it learns a separate weight for every position. An RNN instead reads the steps
**one at a time with the same weights**, so it can handle sequences of **any length** — exactly like a
convolution reuses one kernel at every position of an image (Lab 04), an RNN reuses one set of weights at
every position in **time**.


In [ ]:
a = torch.tensor([1., 2., 3., 4., 5.])
b = torch.flip(a, dims=[0])          # the same numbers, reversed

print("sequence A:", a.tolist(), "  sum =", a.sum().item(), " mean =", a.mean().item())
print("sequence B:", b.tolist(), "  sum =", b.sum().item(), " mean =", b.mean().item())
print()
print("Same numbers. Same sum. Same mean.")
print("But A is rising and B is falling -> the ORDER carries the information.")

## 1.2 Our raw signal: a sine wave

One line of NumPy gives us 400 numbers, sampled at equal time intervals. Think of it as "a sensor
reading, measured 400 times".

We keep it in **`float32`**, the data type PyTorch uses for network weights. (NumPy defaults to
`float64`; mixing the two is one of the most common beginner errors — see the cheat sheet in
[Step 10](#step10).)


In [ ]:
N_POINTS = 400                                    # how many measurements in total
t = np.linspace(0, 40, N_POINTS)                  # "time": 400 equally spaced instants
wave = np.sin(t).astype(np.float32)               # the signal itself  (float32 for PyTorch!)

print("wave shape:", wave.shape, "| dtype:", wave.dtype)
print("first 8 values:", np.round(wave[:8], 3))

plt.figure(figsize=(9, 2.6))
plt.plot(t, wave, lw=1.5)
plt.title("Our whole dataset: one sine wave, 400 points")
plt.xlabel("time"); plt.ylabel("value"); plt.grid(alpha=0.3)
plt.show()

## 1.3 From one long signal to many training examples: **sliding windows**

A network needs many `(input, target)` pairs, but we only have *one* long signal. The standard trick is
a **sliding window**: take `SEQ_LEN` consecutive values as the input, and the *very next* value as the
target. Then slide one step to the right and repeat.

```
wave:   v0 v1 v2 v3 v4 v5 v6 v7 ...            (SEQ_LEN = 4 in this drawing)

example 0:   input = [v0 v1 v2 v3]   ->  target = v4
example 1:   input = [v1 v2 v3 v4]   ->  target = v5
example 2:   input = [v2 v3 v4 v5]   ->  target = v6
                    ...
```

With 400 points and `SEQ_LEN = 20` this gives us **380** training examples out of one wave.


In [ ]:
SEQ_LEN = 20          # how many past values the model may look at


def make_windows(series, seq_len):
    '''Cut a 1-D signal into (window, next_value) pairs.

    Returns:
        X: tensor (n_windows, seq_len, 1)  - the inputs
        y: tensor (n_windows, 1)           - the value that comes right after each window
    '''
    X, y = [], []
    for i in range(len(series) - seq_len):
        X.append(series[i:i + seq_len])      # seq_len values in a row
        y.append(series[i + seq_len])        # the next one
    X = torch.tensor(np.array(X)).unsqueeze(-1)   # (N, seq_len) -> (N, seq_len, 1)
    y = torch.tensor(np.array(y)).unsqueeze(-1)   # (N,)         -> (N, 1)
    return X, y


X, y = make_windows(wave, SEQ_LEN)

print("X shape:", X.shape, "  <- (n_windows, seq_len, input_size)")
print("y shape:", y.shape, "  <- (n_windows, 1)")

## 1.4 The shape rule you must remember

**PyTorch recurrent layers want a 3-D tensor.** With `batch_first=True` (which we always use in this
notebook, and which we strongly recommend) it is:

| Axis | Name | Meaning | Ours |
|:--|:--|:--|:--|
| 0 | `batch` | how many independent sequences at once | 16 windows per batch |
| 1 | `seq_len` | how many time steps in each sequence | 20 |
| 2 | `input_size` | how many numbers arrive **at one time step** | **1** (a single sensor value) |

That last axis confuses everyone at first. `input_size` is **not** the length of the sequence — it is the
size of *one* measurement:

* one temperature per hour → `input_size = 1`
* temperature + humidity + pressure per hour → `input_size = 3`
* one word encoded as a 300-dimensional embedding → `input_size = 300`

> ⚠️ **Careful:** by default `nn.RNN` expects `(seq_len, batch, input_size)` — time first! Always pass
> `batch_first=True` so the batch comes first, like everywhere else in PyTorch.

The next cell prints one concrete example so the numbers stop being abstract.


In [ ]:
print("--- training example 0 ---")
print("input  X[0] (20 time steps, 1 value each), squeezed for readability:")
print(np.round(X[0].squeeze(-1).numpy(), 3))
print("target y[0]:", round(y[0].item(), 3))
print()
print("X[0, 5] =", X[0, 5].tolist(), " <- the value at time step 5 of window 0 (a list of 1 number)")

plt.figure(figsize=(7, 2.8))
plt.plot(range(SEQ_LEN), X[0].squeeze(-1), "o-", label="input window (20 values)")
plt.plot([SEQ_LEN], [y[0].item()], "r*", ms=16, label="target (the next value)")
plt.title("One training example"); plt.xlabel("time step inside the window")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 1.5 Train / test split — **do not shuffle time**

In Lab 02 we split the flowers *randomly*. For a forecasting task that would be **cheating**: a randomly
chosen test window could sit *before* a training window, so the model would be tested on a period it has
effectively already seen. The honest split for time series is **by time**: train on the past, test on the
future.

> 🔑 **Rule of thumb.** Forecasting → split by time. (Shuffling the *windows inside the training set* is
> still fine and even useful, because each window is one independent example — that is what
> `DataLoader(..., shuffle=True)` will do in [Step 5](#step5). Shuffling **inside** a window would destroy
> the sequence itself.)


In [ ]:
N_TRAIN = int(0.8 * len(X))          # first 80% of the windows (the "past")

X_train, y_train = X[:N_TRAIN], y[:N_TRAIN]
X_test,  y_test  = X[N_TRAIN:], y[N_TRAIN:]

print(f"train windows: {len(X_train)}   (from the first part of the wave)")
print(f"test  windows: {len(X_test)}   (from the last part - the 'future')")

split_point = N_TRAIN + SEQ_LEN      # where the test region starts in the original wave
plt.figure(figsize=(9, 2.6))
plt.plot(t[:split_point], wave[:split_point], label="train region", lw=1.5)
plt.plot(t[split_point:], wave[split_point:], label="test region (never seen)", lw=1.5)
plt.axvline(t[split_point], color="k", ls="--", lw=1)
plt.title("Split by time, not at random"); plt.xlabel("time"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
<a id="step2"></a>
# Step 2 · What an RNN Really Does

**Goal:** understand the *one* formula behind every RNN, compute it by hand, then check that
`nn.RNN` gives exactly the same numbers.

## 2.1 One formula, repeated

An RNN keeps a vector called the **hidden state** $h$ — its memory. It starts at zero (it has seen
nothing yet), and at every time step it is **updated** from two things: the new input and its own
previous value.

$$h_t \;=\; \tanh\!\big(W_{xh}\,x_t \;+\; W_{hh}\,h_{t-1} \;+\; b\big)$$

| Symbol | Shape | Meaning |
|:--|:--|:--|
| $x_t$ | `(input_size,)` | the input **at time step _t_** |
| $h_{t-1}$ | `(hidden_size,)` | the memory **after the previous step** |
| $W_{xh}$ | `(hidden_size, input_size)` | how the new input enters the memory |
| $W_{hh}$ | `(hidden_size, hidden_size)` | how the old memory is carried forward |
| $b$ | `(hidden_size,)` | bias |
| $\tanh$ | — | squashes every component into $(-1, 1)$ so the memory cannot explode |

Unrolled over three time steps it looks like this — **the same box, reused**:

```
        x_1            x_2            x_3
         │              │              │
         ▼              ▼              ▼
   ┌───────────┐  ┌───────────┐  ┌───────────┐
h_0│  RNN cell │──│  RNN cell │──│  RNN cell │── h_3 ──▶ Linear ──▶ prediction
──▶│  W_xh,W_hh│h1│  W_xh,W_hh│h2│  W_xh,W_hh│
   └───────────┘  └───────────┘  └───────────┘
        ▲ the SAME weights in all three boxes (weight sharing in time)
```

Two consequences of that picture, worth saying out loud:

1. **The number of parameters does not depend on the sequence length.** A 20-step and a 200-step
   sequence use the same weights — just applied more times.
2. **Everything the model knows about the past must fit in $h$.** That is the RNN's strength (a compact
   summary) and its weakness (a long-ago detail can be overwritten — see [Step 7](#step7)).


### 🧪 Demo — the hidden state, computed by hand

Tiny on purpose: `input_size = 1`, `hidden_size = 2`, a sequence of **3** numbers. Watch the memory
`h` change at every step. This loop *is* an RNN — there is nothing else hidden inside `nn.RNN`.


In [ ]:
# A sequence of 3 time steps, 1 number each: (seq_len, input_size)
x_seq = torch.tensor([[1.0],
                      [0.5],
                      [-1.0]])

# Weights we choose by hand (normally these are learned).
W_xh = torch.tensor([[0.5], [-0.8]])              # (hidden=2, input=1)
W_hh = torch.tensor([[0.1, 0.4], [0.3, -0.5]])    # (hidden=2, hidden=2)
b    = torch.tensor([0.0, 0.1])                   # (hidden=2,)

h = torch.zeros(2)                                # h_0: the memory starts empty
print("h_0 =", [round(v, 4) for v in h.tolist()], " (no memory yet)\n")

for step, x_t in enumerate(x_seq, start=1):
    h = torch.tanh(W_xh @ x_t + W_hh @ h + b)     # <-- THE formula, one line
    print(f"t = {step}:  x_t = {x_t.item():+.2f}  ->  h_{step} = {[round(v, 4) for v in h.tolist()]}")

print("\nThe last h is the model's summary of the whole sequence.")

### 🧪 Demo — the same thing with `nn.RNN`

`nn.RNN` does exactly that loop, in fast C++ code. To prove it, we copy **our** weights into the layer
and compare the numbers.

One detail: PyTorch stores **two** bias vectors (`bias_ih_l0` and `bias_hh_l0`) instead of one, and adds
both. Our single $b$ therefore goes into the first one, and we zero the second.

What comes back are **two** tensors — remember these, they are the source of most RNN bugs:

| Returned | Shape (`batch_first=True`) | What it is |
|:--|:--|:--|
| `output` | `(batch, seq_len, hidden_size)` | the hidden state **at every time step** |
| `h_n` | `(num_layers, batch, hidden_size)` | the hidden state **after the last step only** |

So `output[:, -1, :]` and `h_n[-1]` are **the same numbers** (for a 1-layer, one-direction RNN).


In [ ]:
rnn_demo = nn.RNN(input_size=1, hidden_size=2, batch_first=True)   # tanh is the default

# Copy our hand-made weights into the layer (no_grad: we are editing parameters, not training).
with torch.no_grad():
    rnn_demo.weight_ih_l0.copy_(W_xh)
    rnn_demo.weight_hh_l0.copy_(W_hh)
    rnn_demo.bias_ih_l0.copy_(b)
    rnn_demo.bias_hh_l0.zero_()        # PyTorch adds bias_ih + bias_hh; we put everything in the first

batch = x_seq.unsqueeze(0)             # (3, 1) -> (1, 3, 1) = 1 sequence, 3 steps, 1 feature
output, h_n = rnn_demo(batch)

print("input  shape:", tuple(batch.shape),  " (batch, seq_len, input_size)")
print("output shape:", tuple(output.shape), " (batch, seq_len, hidden_size)  <- h at EVERY step")
print("h_n    shape:", tuple(h_n.shape),    " (num_layers, batch, hidden_size)  <- h at the LAST step")
print()
print("hidden state at each time step (from nn.RNN):")
print(output[0].detach().numpy().round(4))
print()
print("same as our loop?          ", torch.allclose(output[0, -1], h, atol=1e-6))
print("h_n[-1] == output[:,-1,:]? ", torch.allclose(h_n[-1], output[:, -1, :]))

> 🔑 **Key idea.** An RNN is **one small neural network applied again and again**, carrying a memory
> vector from one time step to the next. Training it is the same as always — the loss is back-propagated
> through the *unrolled* chain (this is called *back-propagation through time*), and every copy of the
> weights receives its share of the gradient.

### ✍️ Quick check (30 seconds)

An `nn.RNN(input_size=3, hidden_size=8, batch_first=True)` is given a batch of 32 sequences, each 50
steps long. Before running the cell, write down: (a) the shape of the input, (b) the shape of `output`,
(c) the shape of `h_n`, (d) the number of parameters in the layer.


In [ ]:
layer = nn.RNN(input_size=3, hidden_size=8, batch_first=True)
x_check = torch.randn(32, 50, 3)
out_check, hn_check = layer(x_check)

print("input :", tuple(x_check.shape))
print("output:", tuple(out_check.shape))
print("h_n   :", tuple(hn_check.shape))
print()
n_params = sum(p.numel() for p in layer.parameters())
print("parameters:", n_params, " = W_xh (8x3=24) + W_hh (8x8=64) + two biases (8+8=16)")
print("note: 50 time steps, but the parameter count does not mention 50 at all.")

---
<a id="step3"></a>
# Step 3 · The Model

**Goal:** wrap `nn.RNN` in an `nn.Module`, exactly like the classifiers of Lab 02.

Our task is **many-to-one**: many inputs (20 time steps) → one output (the next value).

```
   window of 20 values  ──▶  nn.RNN  ──▶  h after the last step  ──▶  nn.Linear  ──▶  1 number
      (20, 1)                            (hidden_size,)                              prediction
```

So the model is just **two layers**:

| Layer | Job | Shape in → out |
|:--|:--|:--|
| `nn.RNN(1, 16, batch_first=True)` | read the sequence, keep a memory of 16 numbers | `(B, 20, 1)` → `(B, 20, 16)` |
| `nn.Linear(16, 1)` | turn the final memory into one prediction | `(B, 16)` → `(B, 1)` |

The only genuinely new line compared to Lab 02 is `out[:, -1, :]`: **take the last time step**. That is
where we say "use the memory after the model has read the whole window".

> ❓ **Why the last step and not the average of all of them?** Because $h_T$ is the only state that has
> seen *every* value in the window. Averaging is a legitimate alternative (try it in
> [Exercise 3](#step9)), but "take the last hidden state" is the standard many-to-one recipe.


In [ ]:
class SimpleRNN(nn.Module):
    '''Read a sequence with one RNN layer, then predict one number from the final hidden state.'''

    def __init__(self, input_size=1, hidden_size=16, output_size=1):
        super().__init__()                         # ALWAYS first (Lab 02, step 3)
        self.rnn = nn.RNN(
            input_size=input_size,                 # numbers per time step (1 sensor value)
            hidden_size=hidden_size,               # size of the memory
            batch_first=True,                      # -> (batch, seq_len, input_size)
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):                          # x: (batch, seq_len, input_size)
        out, h_n = self.rnn(x)                     # out: (batch, seq_len, hidden) | h_n: (1, batch, hidden)
        last_hidden = out[:, -1, :]                # (batch, hidden)  <- memory after the LAST time step
        return self.fc(last_hidden)                # (batch, 1)       <- one prediction per sequence


model = SimpleRNN(input_size=1, hidden_size=16, output_size=1).to(DEVICE)
print(model)
print("\ntrainable parameters:", sum(p.numel() for p in model.parameters()))

### 🧪 Sanity check before training — *always do this*

Push one small batch through the untrained model and check the **shapes**. Ninety percent of RNN bugs
are shape bugs, and they are much easier to find here than inside a training loop.


In [ ]:
demo_batch = X_train[:4].to(DEVICE)         # 4 windows
with torch.no_grad():                       # no gradients needed for a shape check
    demo_pred = model(demo_batch)

print("input  :", tuple(demo_batch.shape), " (4 windows, 20 steps, 1 value)")
print("output :", tuple(demo_pred.shape),  " (4 predictions, 1 number each)  ✅")
print()
print("untrained predictions:", demo_pred.squeeze(-1).cpu().numpy().round(3))
print("true next values    :", y_train[:4].squeeze(-1).numpy().round(3))
print("\nRandom weights -> nonsense predictions. That is expected; training comes next.")

---
<a id="step4"></a>
# Step 4 · Loss Function and Optimizer

**Goal:** say *how wrong* a prediction is, and *how* to improve the weights. Identical to Lab 02 —
only the loss changes, because our target is a **number**, not a class.

| Task | Target | Loss | Output layer |
|:--|:--|:--|:--|
| **Regression** (this notebook) | a real number | `nn.MSELoss()` | `nn.Linear(h, 1)`, no activation |
| Classification ([Step 8](#step8)) | a class index | `nn.CrossEntropyLoss()` | `nn.Linear(h, n_classes)`, raw logits |

**MSE** = mean of $(\text{prediction} - \text{target})^2$. Squaring makes every error positive and
punishes big mistakes much harder than small ones.

**Adam** is our usual optimizer: `model.parameters()` is the wiring between the thing being optimised
(the model) and the thing doing the optimising. `lr=0.01` is a good starting point for a network this
small.


In [ ]:
criterion = nn.MSELoss()                                   # regression loss
optimizer = optim.Adam(model.parameters(), lr=0.01)        # the learning rule

print("loss     :", criterion)
print("optimizer:", optimizer.__class__.__name__, "| lr =", optimizer.param_groups[0]["lr"])

# A useful reference number: the loss of a model that always predicts the mean of the training targets.
baseline = criterion(torch.full_like(y_test, y_train.mean().item()), y_test).item()
print(f"\nBaseline MSE (always predict the training mean): {baseline:.4f}")
print("Our model must get clearly below this to be worth anything.")

---
<a id="step5"></a>
# Step 5 · The Training Loop

**Goal:** turn random weights into a working forecaster.

This is **the same loop you already know**. Nothing about it is RNN-specific — that is the whole point
of the PyTorch recipe.

```python
optimizer.zero_grad()    # 1. clear the gradients of the previous step
loss.backward()          # 2. back-propagate (here: through time)
optimizer.step()         # 3. update every weight
```

First we wrap the training windows in a `DataLoader` so they are served in shuffled mini-batches of 16.
Shuffling the **windows** is fine (each window is an independent example); the time steps *inside* a
window are of course never touched.


In [ ]:
BATCH_SIZE = 16
EPOCHS = 60

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=BATCH_SIZE,
                          shuffle=True)      # shuffle the WINDOWS, never inside a window

print(f"{len(X_train)} training windows -> {len(train_loader)} batches of up to {BATCH_SIZE}")
xb, yb = next(iter(train_loader))
print("one batch:", tuple(xb.shape), "->", tuple(yb.shape))

In [ ]:
@torch.no_grad()                                   # no gradients while evaluating
def mse_on(model, X_data, y_data):
    '''Mean squared error of the model over a whole tensor dataset.'''
    model.eval()                                   # evaluation mode
    pred = model(X_data.to(DEVICE))
    return criterion(pred, y_data.to(DEVICE)).item()


train_losses, test_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()                                  # training mode
    running_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)      # model and data on the same device

        optimizer.zero_grad()                      # 1. clear old gradients
        pred = model(xb)                           #    FORWARD  (batch, 1)
        loss = criterion(pred, yb)                 #    how wrong are we?
        loss.backward()                            # 2. BACKWARD through time
        optimizer.step()                           # 3. update the weights

        running_loss += loss.item() * xb.size(0)   # sum, weighted by batch size

    train_losses.append(running_loss / len(X_train))
    test_losses.append(mse_on(model, X_test, y_test))

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:3d} | train MSE {train_losses[-1]:.6f} | test MSE {test_losses[-1]:.6f}")

print("\nTraining finished ✅")

### The learning curve

Loss on a **log scale**, because it falls by orders of magnitude. What a healthy curve looks like:

* both lines fall fast at the start, then flatten;
* the **test** line stays close to the training line → no overfitting (our model is small and the data is
  clean and periodic);
* if the test line turned upward while the train line kept falling, that would be **overfitting**.


In [ ]:
plt.figure(figsize=(6.5, 3.2))
plt.plot(train_losses, label="train")
plt.plot(test_losses, label="test")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("MSE (log scale)")
plt.title("Learning curve"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f"final train MSE: {train_losses[-1]:.6f}")
print(f"final test  MSE: {test_losses[-1]:.6f}")
print(f"baseline    MSE: {baseline:.6f}   <- we should be far below this")

---
<a id="step6"></a>
# Step 6 · Using the Trained Model

**Goal:** actually *use* the network — this is the part students usually miss.

There are **two different ways** to use a sequence model, and they are not equally easy:

| Mode | What you feed in | How hard |
|:--|:--|:--|
| **One-step prediction** | 20 **real** values → predict the next one | easy — every input is ground truth |
| **Multi-step forecast** | predict, then feed *your own prediction* back in, repeatedly | hard — errors accumulate |

## 6.1 One-step prediction on the test region

For every test window we predict the next value and plot it on top of the truth. The two curves should
sit almost exactly on top of each other.

Note `model.eval()` and `torch.no_grad()`: evaluation mode, and no gradient bookkeeping (faster, less
memory).


In [ ]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test.to(DEVICE)).cpu()

time_axis = t[N_TRAIN + SEQ_LEN:]        # where these predictions live on the original time axis

plt.figure(figsize=(9, 3))
plt.plot(time_axis, y_test.squeeze(-1), label="true", lw=2)
plt.plot(time_axis, y_pred_test.squeeze(-1), "--", label="predicted", lw=2)
plt.title("One-step-ahead prediction on the unseen test region")
plt.xlabel("time"); plt.ylabel("value"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

errors = (y_pred_test - y_test).squeeze(-1)
print(f"mean absolute error: {errors.abs().mean():.4f}")
print(f"largest error      : {errors.abs().max():.4f}")

## 6.2 Multi-step forecasting — predicting the future from the future

Now the interesting one. We give the model **one** real window and then ask it to keep going on its own:

```
window = [v1 ... v20]        -> predict p21
window = [v2 ... v20, p21]   -> predict p22        (p21 is now an INPUT)
window = [v3 ... v20, p21, p22] -> predict p23
...
```

This is called **free-running** or **autoregressive** generation, and it is how language models write
text one word at a time. The catch: from step 2 onwards the model eats **its own mistakes**, so small
errors compound.

Below we seed the model with the **last 20 real values** of our wave and let it invent the next 150 —
values that lie completely outside the dataset. Since the signal is a sine, we know what the truth
*would* have been, so we can plot both. Watch how long the forecast stays in phase.


In [ ]:
@torch.no_grad()
def forecast(model, seed_window, n_steps):
    '''Predict n_steps values into the future, feeding each prediction back as input.

    Args:
        seed_window: tensor (seq_len, 1) - the last real values we know.
        n_steps: how many steps to predict.
    '''
    model.eval()
    window = seed_window.clone().to(DEVICE)          # (seq_len, 1)
    predictions = []

    for _ in range(n_steps):
        x = window.unsqueeze(0)                      # (1, seq_len, 1) - a batch of one
        next_value = model(x)                        # (1, 1)
        predictions.append(next_value.item())
        # slide the window: drop the oldest value, append our own prediction
        window = torch.cat([window[1:], next_value.view(1, 1)], dim=0)

    return np.array(predictions)


N_FUTURE = 150                                   # predict far beyond the end of our 400 points

# The seed: the last 20 REAL values of the wave. Everything after this is invented by the model.
seed = torch.tensor(wave[-SEQ_LEN:]).unsqueeze(-1)   # (seq_len, 1)
future = forecast(model, seed, N_FUTURE)

# We happen to know the true continuation, because our signal is a sine: sin(t) for future t.
dt = t[1] - t[0]
t_future = t[-1] + dt * np.arange(1, N_FUTURE + 1)
true_future = np.sin(t_future)

plt.figure(figsize=(9.5, 3.2))
plt.plot(t[-60:], wave[-60:], "k", lw=2, label="known data (the seed is its last 20 points)")
plt.plot(t_future, true_future, lw=2, label="true future")
plt.plot(t_future, future, "--", lw=2, label="free-running forecast")
plt.axvline(t[-1], color="gray", ls=":", lw=1)
plt.title(f"{N_FUTURE}-step forecast, generated from a single 20-value seed")
plt.xlabel("time"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

drift = np.abs(future - true_future)
print(f"mean error over the first 20 forecast steps: {drift[:20].mean():.4f}")
print(f"mean error over the last  20 forecast steps: {drift[-20:].mean():.4f}   <- errors accumulate")

## 6.3 A bonus that only an RNN gives you: **any sequence length**

We trained on windows of exactly 20 steps. Because the same weights are applied at every time step,
the very same model happily reads a window of 5 steps or 100 steps — the loop just runs a different
number of times. A feed-forward network trained on 20 inputs would simply **crash**.

(Working on a length it was never trained on is not always *accurate*, but it is legal — and it is why
one RNN can handle sentences of different lengths.)


In [ ]:
for length in [5, 20, 50, 100]:
    x_any = torch.tensor(wave[:length]).view(1, length, 1).to(DEVICE)   # (1, length, 1)
    with torch.no_grad():
        p = model(x_any).item()
    print(f"sequence length {length:3d} -> prediction {p:+.4f}   (true next value {wave[length]:+.4f})")

print("\nSame weights, different lengths, no error. Try that with nn.Linear(20, 1).")

---
<a id="step7"></a>
# Step 7 · LSTM and GRU — a One-Line Upgrade

**Goal:** know what to reach for when the plain RNN is not enough.

## 7.1 The problem with the plain RNN: a short memory

To learn a dependency that spans 100 time steps, the gradient must travel back through 100 multiplications
by the same matrix $W_{hh}$. If those factors are slightly smaller than 1 the gradient **shrinks to
nothing** (*vanishing gradients*); slightly larger and it **blows up**. In practice a plain `nn.RNN`
reliably remembers roughly **10–20 steps**.

**LSTM** (Long Short-Term Memory, 1997) and **GRU** (2014) fix this with *gates* — small learned
sigmoids that decide, at every step, what to **keep**, what to **forget** and what to **output**. The LSTM
also carries a second vector, the **cell state**, which flows through mostly unchanged — a "conveyor belt"
for long-range information.

| Layer | Parameters (per unit) | Memory | When to use |
|:--|:--|:--|:--|
| `nn.RNN` | 1× | short | short sequences, teaching, baselines |
| `nn.GRU` | 3× | long | the usual practical default — nearly LSTM quality, cheaper |
| `nn.LSTM` | 4× | long | long sequences, the classic safe choice |

**The good news:** they are drop-in replacements. Same constructor arguments, same `batch_first`, same
input shape. The *only* difference in your code is what the layer returns:

```python
out, h_n          = rnn(x)     # RNN and GRU
out, (h_n, c_n)   = lstm(x)    # LSTM also returns the cell state
```

Writing `out, _ = self.rec(x)` covers all three at once.


In [ ]:
class SeqModel(nn.Module):
    '''The same model as SimpleRNN, but the recurrent layer is chosen by name.'''

    def __init__(self, kind="rnn", input_size=1, hidden_size=16, output_size=1):
        super().__init__()
        layer_class = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rec = layer_class(input_size, hidden_size, batch_first=True)   # <- the one line that changes
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rec(x)              # "_" swallows h_n (RNN/GRU) or (h_n, c_n) (LSTM)
        return self.fc(out[:, -1, :])     # last time step -> prediction


for kind in ["rnn", "gru", "lstm"]:
    m = SeqModel(kind, hidden_size=16)
    print(f"{kind.upper():5s}: {sum(p.numel() for p in m.parameters()):5d} parameters")

## 7.2 The training loop, packaged for reuse

We are about to train several models, so we put the loop of [Step 5](#step5) into a function. It is
**exactly the same code** — read it once and confirm that nothing new happens here.


In [ ]:
def train_model(model, X_tr, y_tr, X_te, y_te, epochs=60, lr=0.01, batch_size=16,
                criterion=nn.MSELoss(), verbose=False):
    '''Train any model on tensor data and return (train_losses, test_losses).'''
    model = model.to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    tr_hist, te_hist = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
        tr_hist.append(total / len(X_tr))

        model.eval()
        with torch.no_grad():
            te_hist.append(criterion(model(X_te.to(DEVICE)), y_te.to(DEVICE)).item())

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"  epoch {epoch:3d} | train {tr_hist[-1]:.6f} | test {te_hist[-1]:.6f}")

    return tr_hist, te_hist


print("train_model() ready - the same loop as Step 5, now reusable")

### 🧪 Demo — the three layers on our sine wave

On a task this easy all three will do well; the point is that **swapping them costs one word**. The
difference shows up on the harder task in 7.3.


In [ ]:
plt.figure(figsize=(6.5, 3.2))
for kind in ["rnn", "gru", "lstm"]:
    torch.manual_seed(SEED)                                   # same start for a fair comparison
    m = SeqModel(kind, hidden_size=16)
    tr, te = train_model(m, X_train, y_train, X_test, y_test, epochs=60)
    plt.plot(te, label=f"{kind.upper()}  (final test MSE {te[-1]:.5f})")
    print(f"{kind.upper():5s} final test MSE: {te[-1]:.6f}")

plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("test MSE (log)")
plt.title("Same data, same loop, one word changed"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7.3 *Why* the plain RNN forgets — look at the gradients

Here is the vanishing-gradient problem, measured instead of asserted. We feed a 60-step sequence to an
RNN and ask, for every time step *t*:

> how much does the **final** hidden state $h_{60}$ change if I nudge the input $x_t$?

That is exactly $|\partial h_{60} / \partial x_t|$, and PyTorch hands it to us with one `backward()` on an
input tensor created with `requires_grad=True`. It is also the size of the learning signal that reaches
step *t*: small here means "this step cannot be taught anything".


In [ ]:
def gradient_profile(layer, seq_len=60, batch=64):
    '''How strongly does the LAST hidden state depend on the input at each time step?'''
    x = torch.randn(batch, seq_len, 1, requires_grad=True)   # we want gradients w.r.t. the INPUT
    out, _ = layer(x)
    out[:, -1, :].sum().backward()                           # differentiate the final hidden state
    return x.grad.abs().mean(dim=(0, 2))                     # (seq_len,) averaged over batch and features


torch.manual_seed(SEED)
profile = gradient_profile(nn.RNN(1, 32, batch_first=True), seq_len=60)

plt.figure(figsize=(6.5, 3.2))
plt.semilogy(range(1, 61), profile.numpy())
plt.xlabel("time step t"); plt.ylabel("|d h_60 / d x_t|   (log scale)")
plt.title("How much of each input survives in the final hidden state"); plt.grid(alpha=0.3)
plt.show()

print(f"influence of the LAST  input (t = 60): {profile[-1]:.2e}")
print(f"influence of the FIRST input (t =  1): {profile[0]:.2e}")
print(f"the first input matters {(profile[-1] / profile[0]):.1e} times less than the last one")

> 🔑 **Read the numbers.** On a log axis the curve is almost a straight line: the influence of an input
> decays **exponentially** with how long ago it arrived, and step 1 is many orders of magnitude weaker
> than step 60. Each step multiplies the signal by $W_{hh}$ and by $\tanh'(\cdot) \le 1$ — sixty times
> over. Whatever the network should learn about step 1 arrives as a gradient of essentially zero:
> **vanishing gradients**.
>
> Gates do not cancel this arithmetic, but they make long memory **learnable**: an LSTM can *learn* to
> hold its forget gate near 1, which carries a value through the cell state almost unchanged instead of
> shrinking it at every step.

## 7.4 A task that needs a long memory: **the adding problem**

The classic benchmark from the LSTM papers, and easy to state:

> Every time step carries **two** numbers: a random **value** in $[0, 1)$ and a **flag** that is 0 or 1.
> Exactly **two** steps of the sequence are flagged. **Output the sum of the two flagged values.**

To solve it the network must spot a flag, *hold that value* while an arbitrary number of irrelevant steps
go past, and add it to the second one. Note that this is also our first dataset with `input_size = 2` —
exactly the "several numbers per time step" case from [Step 1.4](#step1).

Always know the score to beat: the target is a sum of two uniform values, so always predicting its mean
(1.0) gives an MSE of about **0.16**. A model sitting at 0.16 has learned nothing.


In [ ]:
def make_adding_task(n_samples, seq_len, seed=0):
    '''Each step = (value, flag). Exactly two steps are flagged; the target is the sum of their values.'''
    g = torch.Generator().manual_seed(seed)
    values = torch.rand(n_samples, seq_len, 1, generator=g)          # random numbers in [0, 1)
    flags = torch.zeros(n_samples, seq_len, 1)
    for i in range(n_samples):
        marked = torch.randperm(seq_len, generator=g)[:2]            # two random positions
        flags[i, marked, 0] = 1.0
    targets = (values * flags).sum(dim=1)                            # (n_samples, 1)
    return torch.cat([values, flags], dim=2), targets                # (n, seq_len, 2), (n, 1)


ADD_LEN = 30
Xa_tr, ya_tr = make_adding_task(2000, ADD_LEN, seed=1)
Xa_te, ya_te = make_adding_task(500, ADD_LEN, seed=2)

print("X:", tuple(Xa_tr.shape), " <- input_size = 2 now: (value, flag) at every step")
print("y:", tuple(ya_tr.shape))
print()

# Look at one example whose two flags are far apart - that is where a long memory is needed.
positions = [(Xa_tr[i, :, 1] == 1).nonzero().flatten().tolist() for i in range(len(Xa_tr))]
i = max(range(len(positions)), key=lambda k: positions[k][1] - positions[k][0])
first, second = positions[i]

np.set_printoptions(linewidth=120)
print(f"example {i}")
print("  values:", Xa_tr[i, :, 0].numpy().round(2))
print("  flags :", Xa_tr[i, :, 1].int().numpy())
print(f"  flagged positions {first} and {second} ({second - first} steps apart)")
print(f"  target = {Xa_tr[i, first, 0]:.2f} + {Xa_tr[i, second, 0]:.2f} = {ya_tr[i].item():.2f}")
print("  -> the value at the first flag must survive in the memory until the second one arrives")
print()
print(f"'learned nothing' MSE (always predict the mean): {((ya_te - ya_tr.mean()) ** 2).mean():.3f}")

In [ ]:
add_results = {}
for kind in ["rnn", "lstm", "gru"]:
    torch.manual_seed(SEED)                                   # same starting point for all three
    m = SeqModel(kind, input_size=2, hidden_size=32)
    tr, te = train_model(m, Xa_tr, ya_tr, Xa_te, ya_te, epochs=60, lr=0.005, batch_size=64)
    add_results[kind] = te
    print(f"{kind.upper():5s} final test MSE: {te[-1]:.4f}")

plt.figure(figsize=(6.5, 3.2))
for kind, te in add_results.items():
    plt.plot(te, label=kind.upper())
plt.axhline(((ya_te - ya_tr.mean()) ** 2).mean().item(), color="gray", ls="--", label="learned nothing")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("test MSE (log)")
plt.title(f"The adding problem, sequence length {ADD_LEN}"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

> 🔑 **Read the plot.** The plain **RNN** stays stuck near the "learned nothing" line — it cannot hold the
> first flagged value long enough to add the second. The **LSTM** and **GRU** fall by two or three orders
> of magnitude: their gates learn to keep that value and ignore the distractors. This is the experiment
> that made gated networks the default, and the only thing that changed in your code was the layer's name.
>
> Exact numbers vary between runs; the *ordering* is the robust part. Want it harder? Set `ADD_LEN = 60`
> and re-run — the plain RNN gets worse, the gated ones cope for much longer.
>
> ⚠️ **Do not over-generalise, though:** on the sine wave in 7.2 the plain RNN was perfectly fine, and
> gates cost 3–4× the parameters and computation. Reach for them when the dependency is **long**.


---
<a id="step8"></a>
# Step 8 · A Second Example — Sequence **Classification**

**Goal:** use the same building blocks for the other big sequence task: give the **whole sequence one
label**. (This is what sentiment analysis does: a sentence in, "positive"/"negative" out.)

## 8.1 The toy dataset: rising or falling?

Each example is 12 numbers. Class **0 = rising**, class **1 = falling**. We build the falling ones by
**reversing** the rising ones, which makes the lesson sharp:

> The two classes contain **exactly the same numbers**. A model that ignores order cannot beat 50 %.
> Only the direction of travel separates them — and that is a property of the *sequence*, not of the
> values.

Only **three things change** compared with our forecasting model:

| | Regression (Steps 3–6) | Classification (here) |
|:--|:--|:--|
| final layer | `nn.Linear(hidden, 1)` | `nn.Linear(hidden, 2)` — one score per class |
| target dtype/shape | `float`, `(N, 1)` | `long`, `(N,)` — class indices |
| loss | `nn.MSELoss()` | `nn.CrossEntropyLoss()` |

The RNN itself, the training loop and the `(batch, seq_len, input_size)` shape are **untouched**.


In [ ]:
def make_updown_dataset(n_samples, seq_len=12, seed=0):
    '''Half rising sequences (label 0), half falling ones (label 1, = a rising one reversed).'''
    g = torch.Generator().manual_seed(seed)
    start = torch.rand(n_samples, 1, generator=g) * 2 - 1            # random starting level
    steps = torch.rand(n_samples, seq_len, generator=g) * 0.2 + 0.05 # positive increments
    seqs = start + torch.cumsum(steps, dim=1)                        # -> rising sequences

    labels = torch.randint(0, 2, (n_samples,), generator=g)          # 0 = rising, 1 = falling
    seqs[labels == 1] = torch.flip(seqs[labels == 1], dims=[1])      # reverse half of them
    seqs = seqs + 0.02 * torch.randn(seqs.shape, generator=g)        # a little noise

    return seqs.unsqueeze(-1), labels                                # (N, seq_len, 1), (N,)


Xc_train, yc_train = make_updown_dataset(800, seed=10)
Xc_test,  yc_test  = make_updown_dataset(200, seed=11)

print("X:", tuple(Xc_train.shape), " (batch, seq_len, input_size) - the same rule as before")
print("y:", tuple(yc_train.shape), " dtype:", yc_train.dtype, " <- long, class indices 0/1")
print("class balance in the training set:", torch.bincount(yc_train).tolist())

plt.figure(figsize=(7, 3))
for i in range(6):
    plt.plot(Xc_train[i, :, 0], "o-", ms=3, label=f"class {yc_train[i].item()}")
plt.title("Six examples: class 0 = rising, class 1 = falling")
plt.xlabel("time step"); plt.legend(fontsize=8, ncol=3); plt.grid(alpha=0.3)
plt.show()

## 8.2 Train it — the same loop, a different loss

`SeqModel(..., output_size=2)` now produces **two scores (logits)** per sequence, and
`nn.CrossEntropyLoss` compares them with the correct class index. Do **not** add a softmax in the model:
PyTorch's cross-entropy applies it internally (Lab 02, Step 4).


In [ ]:
torch.manual_seed(SEED)
clf = SeqModel(kind="rnn", input_size=1, hidden_size=16, output_size=2)

tr_hist, te_hist = train_model(clf, Xc_train, yc_train, Xc_test, yc_test,
                               epochs=30, lr=0.01, batch_size=32,
                               criterion=nn.CrossEntropyLoss(), verbose=True)

# --- accuracy: the class with the highest score wins ---
clf.eval()
with torch.no_grad():
    logits = clf(Xc_test.to(DEVICE))               # (200, 2) raw scores
    predicted = logits.argmax(dim=1).cpu()         # (200,)   the winning class
accuracy = (predicted == yc_test).float().mean().item()

print(f"\ntest accuracy: {accuracy * 100:.1f}%   (50% = random guessing)")
print("first 10 predictions:", predicted[:10].tolist())
print("first 10 true labels:", yc_test[:10].tolist())

### 🧪 Demo — proof that the model uses the *order*

Take a test sequence the model classifies correctly, **reverse it**, and classify again. A model that
only looked at the values (their mean, their spread, their set) could not change its answer. Ours
flips it — because reversing is exactly what turns one class into the other.


In [ ]:
clf.eval()
with torch.no_grad():
    one = Xc_test[:1].to(DEVICE)                        # (1, 12, 1)
    reversed_one = torch.flip(one, dims=[1])            # same values, opposite order

    p_original = clf(one).softmax(dim=1)[0]
    p_reversed = clf(reversed_one).softmax(dim=1)[0]

print("true label of this sequence:", yc_test[0].item())
print(f"original  -> P(rising) = {p_original[0]:.3f}, P(falling) = {p_original[1]:.3f}")
print(f"reversed  -> P(rising) = {p_reversed[0]:.3f}, P(falling) = {p_reversed[1]:.3f}")
print()
print("same 12 numbers, same mean, same min/max - opposite answer. The order is the signal. ✅")

---
<a id="step9"></a>
# Step 9 · Exercises

Do them in order; each one is a small edit of a cell above. Try first, **then** open the solution.

### ✍️ Exercise 1 — How much past does the model need?

Re-train the sine-wave model with `SEQ_LEN` = 3, 5, 10, 20 and 40 (rebuild the windows and the split
each time), and plot the final test MSE against `SEQ_LEN`. Is more history always better?


<details>
<summary>✅ <b>Show solution</b></summary>

```python
results = {}
for L in [3, 5, 10, 20, 40]:
    Xl, yl = make_windows(wave, L)
    n_tr = int(0.8 * len(Xl))
    torch.manual_seed(SEED)
    m = SeqModel("rnn", hidden_size=16)
    _, te = train_model(m, Xl[:n_tr], yl[:n_tr], Xl[n_tr:], yl[n_tr:], epochs=60)
    results[L] = te[-1]
    print(f"SEQ_LEN {L:3d} -> test MSE {te[-1]:.6f}")

plt.figure(figsize=(5.5, 3))
plt.plot(list(results), list(results.values()), "o-")
plt.xlabel("SEQ_LEN"); plt.ylabel("final test MSE"); plt.yscale("log"); plt.grid(alpha=0.3)
plt.show()
```

**What to notice.** Very short windows (3–5) already work quite well — a sine wave is locally very
predictable. Beyond ~20 there is little to gain and training gets slower: a longer window means more
unrolled steps, more computation, and a harder optimisation problem. *More history is not automatically
better.*
</details>


### ✍️ Exercise 2 — Noise, and what "good" means

Add noise to the signal: `noisy = wave + 0.1 * np.random.randn(N_POINTS).astype(np.float32)`. Rebuild
the windows, re-train, and plot predictions against the noisy truth. Why can the test MSE never reach 0
now? What value *should* it approach?


<details>
<summary>✅ <b>Show solution</b></summary>

```python
rng = np.random.RandomState(0)
noisy = (wave + 0.1 * rng.randn(N_POINTS)).astype(np.float32)
Xn, yn = make_windows(noisy, SEQ_LEN)
n_tr = int(0.8 * len(Xn))
torch.manual_seed(SEED)
m = SeqModel("rnn", hidden_size=16)
_, te = train_model(m, Xn[:n_tr], yn[:n_tr], Xn[n_tr:], yn[n_tr:], epochs=60)
print("test MSE:", round(te[-1], 4), " | noise variance:", 0.1 ** 2)
```

The target now contains a random component that **nothing** in the input can predict. Even a perfect
model would predict the clean sine value and still be wrong by the noise, so the MSE cannot go below the
noise variance $0.1^2 = 0.01$. In practice you land a little above it (≈ 0.015–0.02), because the inputs
are noisy too. Getting *below* the floor would mean the model is memorising the noise (overfitting).
This "irreducible error" is worth remembering whenever a loss refuses to go to zero.
</details>


### ✍️ Exercise 3 — Use every hidden state, not only the last

In `SeqModel.forward` we use `out[:, -1, :]`. Replace it with the **mean over time**,
`out.mean(dim=1)`, and compare the test MSE on the sine task and on the adding problem of
[Step 7.4](#step7) — keep `kind="rnn"` for both, so the recurrent layer is not what changes.
Which task benefits, and why?


<details>
<summary>✅ <b>Show solution</b></summary>

```python
class MeanPoolModel(SeqModel):
    def forward(self, x):
        out, _ = self.rec(x)
        return self.fc(out.mean(dim=1))      # average of all hidden states

runs = {"sine":   (X_train, y_train, X_test, y_test, 1, 60, 0.01,  16),
        "adding": (Xa_tr,   ya_tr,   Xa_te,  ya_te,  2, 60, 0.005, 64)}

for name, (Xtr, ytr, Xte, yte, n_in, ep, lr, bs) in runs.items():
    for cls in [SeqModel, MeanPoolModel]:
        torch.manual_seed(SEED)
        m = cls("rnn", input_size=n_in, hidden_size=32)
        _, te = train_model(m, Xtr, ytr, Xte, yte, epochs=ep, lr=lr, batch_size=bs)
        print(f"{name:7s} {cls.__name__:14s} test MSE {te[-1]:.4f}")
```

On the **sine** task the last hidden state is the right choice: the prediction depends on the most recent
values, and averaging dilutes them with older ones. On the **adding problem** averaging helps the plain
RNN, because the states from the early steps — the ones that still remember a flagged value — are read
directly instead of having to survive all the remaining updates.

That is exactly the intuition behind **attention**: instead of squeezing the whole past into one final
state, look back at *all* the states — and, better than a flat average, learn *which* ones matter.
</details>


### ✍️ Exercise 4 — Two features instead of one

Feed the model **both** $\sin(t)$ and $\cos(t)$ at every time step (`input_size=2`) and predict the next
$\sin(t)$. Hint: build `series = np.stack([np.sin(t), np.cos(t)], axis=1)` and write a `make_windows_2d`
that keeps the feature axis. Does the extra channel help?


<details>
<summary>✅ <b>Show solution</b></summary>

```python
series = np.stack([np.sin(t), np.cos(t)], axis=1).astype(np.float32)   # (400, 2)

def make_windows_2d(series, seq_len, target_col=0):
    X = torch.stack([torch.tensor(series[i:i + seq_len]) for i in range(len(series) - seq_len)])
    y = torch.tensor(series[seq_len:, target_col]).unsqueeze(-1)
    return X, y                                    # (N, seq_len, 2), (N, 1)

X2, y2 = make_windows_2d(series, SEQ_LEN)
n_tr = int(0.8 * len(X2))
torch.manual_seed(SEED)
m = SeqModel("rnn", input_size=2, hidden_size=16)
_, te = train_model(m, X2[:n_tr], y2[:n_tr], X2[n_tr:], y2[n_tr:], epochs=60)
print("2-feature test MSE:", round(te[-1], 6))
```

`input_size` is the **only** thing that changes — the loop, the loss and the head are identical.
$(\sin, \cos)$ together give the phase without ambiguity, so this model usually converges faster. This is
exactly how you feed multi-sensor data (temperature + humidity + pressure) to an RNN.
</details>


### ✍️ Exercise 5 — Stack two layers, and go bidirectional

`nn.RNN` takes `num_layers=2` (feed the hidden states of layer 1 into layer 2) and `bidirectional=True`
(read the sequence forwards *and* backwards; the hidden size of `output` doubles). Try both on the
**classification** task of [Step 8](#step8). Then answer: why would `bidirectional=True` be **cheating**
for the forecasting task of Step 6?


<details>
<summary>✅ <b>Show solution</b></summary>

```python
class DeepClassifier(nn.Module):
    def __init__(self, hidden_size=16, num_layers=2, bidirectional=True, n_classes=2):
        super().__init__()
        self.rec = nn.RNN(1, hidden_size, num_layers=num_layers,
                          bidirectional=bidirectional, batch_first=True)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), n_classes)

    def forward(self, x):
        out, _ = self.rec(x)            # (B, T, hidden * num_directions)
        return self.fc(out[:, -1, :])

torch.manual_seed(SEED)
m = DeepClassifier()
train_model(m, Xc_train, yc_train, Xc_test, yc_test, epochs=30,
            criterion=nn.CrossEntropyLoss(), batch_size=32)
m.eval()
with torch.no_grad():
    acc = (m(Xc_test.to(DEVICE)).argmax(1).cpu() == yc_test).float().mean()
print("bidirectional 2-layer accuracy:", round(acc.item() * 100, 1), "%")
```

**Why bidirectional is fine here but not for forecasting.** Classification sees the whole sequence
before it must answer, so reading it backwards too is legitimate. Forecasting must predict step *t+1*
from steps *≤ t*: a backward pass would let information from the future leak into the state — the model
would look great in your notebook and fail completely in production. This class of bug is called
**data leakage**, and it is the most common serious mistake in time-series projects.
</details>


---
<a id="step10"></a>
# Step 10 · Summary, Cheat Sheet, Common Bugs

## What you did

```
1. DATA      one signal  ->  sliding windows  ->  X (batch, seq_len, input_size), y
2. IDEA      h_t = tanh(W_xh x_t + W_hh h_{t-1} + b)     the same weights at every step
3. MODEL     nn.RNN(...)  ->  out[:, -1, :]  ->  nn.Linear(hidden, out)
4. LOSS      MSELoss for numbers | CrossEntropyLoss for classes
5. TRAIN     zero_grad -> forward -> loss -> backward -> step        (unchanged since Lab 02)
6. USE       one-step prediction, and free-running multi-step forecast
7. UPGRADE   nn.RNN -> nn.GRU / nn.LSTM when the memory must be long
8. CLASSIFY  same model, 2 outputs, cross-entropy
```

## 📋 One-page cheat sheet

| Task | Code |
|:--|:--|
| input layout | `(batch, seq_len, input_size)` **with** `batch_first=True` |
| one sequence → a batch of one | `x.unsqueeze(0)` |
| the layer | `nn.RNN(input_size, hidden_size, batch_first=True)` |
| what it returns | `out, h_n = rnn(x)` — `out` is `(B, T, H)`, `h_n` is `(layers, B, H)` |
| LSTM returns | `out, (h_n, c_n) = lstm(x)` — use `out, _ = lstm(x)` to stay generic |
| last hidden state | `out[:, -1, :]` (same as `h_n[-1]` for 1 layer, one direction) |
| many-to-one head | `nn.Linear(hidden_size, n_outputs)` on the last hidden state |
| many-to-many head | `nn.Linear(hidden_size, n_outputs)` applied to **all** of `out` |
| deeper | `num_layers=2` |
| read both directions | `bidirectional=True` → head input is `hidden_size * 2` |
| regression loss | `nn.MSELoss()`, targets `float`, shape `(N, 1)` |
| classification loss | `nn.CrossEntropyLoss()`, targets `long`, shape `(N,)`, **no softmax in the model** |
| stop exploding gradients | `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` before `optimizer.step()` |
| variable-length batches | `nn.utils.rnn.pad_sequence`, then `pack_padded_sequence` |

## 🐞 The five bugs everyone hits

| Symptom | Cause | Fix |
|:--|:--|:--|
| `RuntimeError: input must have 3 dimensions, got 2` | you passed `(batch, seq_len)` | add the feature axis: `x.unsqueeze(-1)` |
| results are nonsense but shapes are fine | forgot `batch_first=True`, so PyTorch read your batch axis as time | pass `batch_first=True` everywhere |
| `expected scalar type Float but found Double` | NumPy `float64` slipped in | `.astype(np.float32)` when creating the data |
| loss stuck at a high value | targets/inputs on very different scales, or `lr` too big/small | standardise the series; try `lr` 0.1 … 0.001 |
| loss becomes `nan` | exploding gradients on a long sequence | gradient clipping, a smaller `lr`, or an LSTM/GRU |

## 🚀 Where this goes next

* **Packing** — real sequences have different lengths; `pack_padded_sequence` lets one batch hold them
  all without the RNN reading the padding.
* **Embeddings** — for text, each word becomes a learned vector: `nn.Embedding(vocab, 300)` feeding an
  `input_size=300` LSTM.
* **Attention & Transformers** — instead of squeezing the past into one hidden state, look at **all**
  previous states and learn which ones matter. Exercise 3 above is the first half of that idea, and it
  is what GPT-style models are built on.

> 🎓 **The one sentence to remember.** An RNN is one small network applied at every time step, passing a
> hidden state — its memory — forward through time; everything else (loss, optimizer, training loop) is
> exactly what you already knew.
